# BM25: взвешенный

Сейчас title+description+params индексируются как одна строка. Пробуем взвесить поля по важности.

In [1]:
import os
import sys

sys.path.append("..")

import bm25s
import pandas as pd

from eval import make_val_split, recall_at_50
from preprocessing import tokenize

train = pd.read_parquet("../data/train.parquet")
train_part, val_queries, val_relevant = make_val_split(train, val_size=0.1, random_state=42)
print("val_queries:", len(val_queries))

val_queries: 35447


Вариант А: повторяем токены поля N раз перед индексацией (работает с одним bm25s-индексом). Токенизируем каждое поле ОТДЕЛЬНО один раз - иначе лемматизация пересчитывалась бы заново на каждую комбинацию весов.

In [2]:
items_corpus = train.drop_duplicates("item_id")[
    ["item_id", "item_title_raw", "item_description_raw", "item_infm_params_text"]
].reset_index(drop=True)
corpus_item_ids = items_corpus["item_id"].tolist()

title_tokens = items_corpus["item_title_raw"].apply(tokenize)
desc_tokens = items_corpus["item_description_raw"].apply(tokenize)
params_tokens = items_corpus["item_infm_params_text"].apply(tokenize)
val_tokens = val_queries["search_query"].apply(tokenize).tolist()
print("поля токенизированы")

поля токенизированы


In [3]:
def eval_weights(w_title, w_desc, w_params):
    cache = f"../data/bm25s_index_weighted/w{w_title}{w_desc}{w_params}"
    if os.path.exists(cache):
        retriever = bm25s.BM25.load(cache, load_corpus=False)
    else:
        combined = [
            t * w_title + d * w_desc + p * w_params
            for t, d, p in zip(title_tokens, desc_tokens, params_tokens)
        ]
        retriever = bm25s.BM25()
        retriever.index(combined, show_progress=False)
        os.makedirs(os.path.dirname(cache), exist_ok=True)
        retriever.save(cache)
    results, _ = retriever.retrieve(val_tokens, corpus=corpus_item_ids, k=50, show_progress=False, n_threads=0)
    predictions = {cid: list(results[i]) for i, cid in enumerate(val_queries["context_id"])}
    return recall_at_50(predictions, val_relevant)


weight_combos = [(1, 1, 1), (3, 2, 1), (3, 1, 1), (2, 2, 1)]
results_table = []
for w in weight_combos:
    r = eval_weights(*w)
    results_table.append((w, r))
    print(f"title:desc:params = {w[0]}:{w[1]}:{w[2]} -> Recall@50 = {r:.4f}")

title:desc:params = 1:1:1 -> Recall@50 = 0.1861


title:desc:params = 3:2:1 -> Recall@50 = 0.1986


title:desc:params = 3:1:1 -> Recall@50 = 0.2037


title:desc:params = 2:2:1 -> Recall@50 = 0.1923


In [4]:
best = max(results_table, key=lambda x: x[1])
baseline = next(r for w, r in results_table if w == (1, 1, 1))
print(f"лучший вариант: {best[0]}, recall={best[1]:.4f}")
print(f"прирост над бейзлайном 1:1:1 ({baseline:.4f}): {best[1] - baseline:+.4f}")

лучший вариант: (3, 1, 1), recall=0.2037
прирост над бейзлайном 1:1:1 (0.1861): +0.0176


## Вариант Б: отдельные индексы + взвешенная сумма скоров

Три независимых BM25-индекса (title/description/params), берём top-200 с каждого, комбинируем взвешенной суммой скоров.

In [5]:
def build_field_index(tokens, cache):
    if os.path.exists(cache):
        return bm25s.BM25.load(cache, load_corpus=False)
    retriever = bm25s.BM25()
    retriever.index(tokens.tolist(), show_progress=False)
    os.makedirs(cache, exist_ok=True)
    retriever.save(cache)
    return retriever


retriever_title = build_field_index(title_tokens, "../data/bm25s_index_title")
retriever_desc = build_field_index(desc_tokens, "../data/bm25s_index_desc")
retriever_params = build_field_index(params_tokens, "../data/bm25s_index_params")

K_FIELD = 200
title_results, title_scores = retriever_title.retrieve(val_tokens, corpus=corpus_item_ids, k=K_FIELD, show_progress=False, n_threads=0)
desc_results, desc_scores = retriever_desc.retrieve(val_tokens, corpus=corpus_item_ids, k=K_FIELD, show_progress=False, n_threads=0)
params_results, params_scores = retriever_params.retrieve(val_tokens, corpus=corpus_item_ids, k=K_FIELD, show_progress=False, n_threads=0)
print("отдельные индексы построены/загружены")

отдельные индексы построены/загружены


In [6]:
def eval_weighted_sum(w_title, w_desc, w_params, top_n=50):
    predictions = {}
    for i, cid in enumerate(val_queries["context_id"]):
        combined = {}
        for iid, s in zip(title_results[i], title_scores[i]):
            combined[iid] = combined.get(iid, 0) + w_title * float(s)
        for iid, s in zip(desc_results[i], desc_scores[i]):
            combined[iid] = combined.get(iid, 0) + w_desc * float(s)
        for iid, s in zip(params_results[i], params_scores[i]):
            combined[iid] = combined.get(iid, 0) + w_params * float(s)
        ranked = sorted(combined.items(), key=lambda x: -x[1])[:top_n]
        predictions[cid] = [iid for iid, _ in ranked]
    return recall_at_50(predictions, val_relevant)


results_table_b = []
for w in weight_combos:
    r = eval_weighted_sum(*w)
    results_table_b.append((w, r))
    print(f"[Б] title:desc:params = {w[0]}:{w[1]}:{w[2]} -> Recall@50 = {r:.4f}")

[Б] title:desc:params = 1:1:1 -> Recall@50 = 0.1891


[Б] title:desc:params = 3:2:1 -> Recall@50 = 0.1982


[Б] title:desc:params = 3:1:1 -> Recall@50 = 0.1929


[Б] title:desc:params = 2:2:1 -> Recall@50 = 0.1937


## А vs Б vs бейзлайн

In [7]:
best_b = max(results_table_b, key=lambda x: x[1])
print(f"бейзлайн (1:1:1, один индекс): {baseline:.4f}")
print(f"Вариант А лучший {best[0]}: {best[1]:.4f} ({best[1] - baseline:+.4f})")
print(f"Вариант Б лучший {best_b[0]}: {best_b[1]:.4f} ({best_b[1] - baseline:+.4f})")

бейзлайн (1:1:1, один индекс): 0.1861
Вариант А лучший (3, 1, 1): 0.2037 (+0.0176)
Вариант Б лучший (3, 2, 1): 0.1982 (+0.0121)


кайфик, используем А